In [ ]:
"""
Catalog-Based Price Scraper
Uses product titles/descriptions from catalog_content to find and scrape prices

⚠️  WARNING: This scrapes TEST data - DO NOT use for training!
"""

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import os
import re
from urllib.parse import quote_plus

# ============================================================================
# CONFIGURATION
# ============================================================================
TEST_CSV = "student_resource/dataset/test.csv"
OUTPUT_DIR = "student_resource/scraped_test_data"
MAX_SAMPLES = 20  # Start with 20 samples
USE_AMAZON_SEARCH = True  # Direct Amazon search (faster, more reliable)
# ============================================================================

USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
HEADERS = {
    'User-Agent': USER_AGENT,
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
    'Accept-Encoding': 'gzip, deflate',
    'Connection': 'keep-alive',
}


def clean_product_title(title):
    """Clean product title for better search results"""
    if not title or pd.isna(title):
        return None
    
    # Convert to string and limit length
    title = str(title)[:150]
    
    # Remove special characters but keep important ones
    title = re.sub(r'[^\w\s\-.,&()]', ' ', title)
    
    # Remove extra whitespace
    title = ' '.join(title.split())
    
    return title.strip()


def search_amazon_product(product_title):
    """Search Amazon for product and return first result URL"""
    try:
        clean_title = clean_product_title(product_title)
        if not clean_title:
            return None
        
        # Take first 80 chars for search (more focused results)
        search_query = clean_title[:80]
        
        # Build Amazon search URL
        search_url = f"https://www.amazon.com/s?k={quote_plus(search_query)}"
        
        print(f"    Searching: {search_query[:50]}...")
        
        # Random delay to be respectful
        time.sleep(random.uniform(2, 4))
        
        response = requests.get(search_url, headers=HEADERS, timeout=15)
        
        if response.status_code != 200:
            print(f"    ✗ Search failed: HTTP {response.status_code}")
            return None
        
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Find product results
        products = soup.find_all('div', {'data-component-type': 's-search-result'})
        
        if not products:
            print("    ✗ No products found")
            return None
        
        # Get first result
        first_product = products[0]
        
        # Find product link
        link = first_product.find('a', {'class': 'a-link-normal s-no-outline'})
        if not link or 'href' not in link.attrs:
            link = first_product.find('a', {'class': 'a-link-normal s-underline-text'})
        
        if link and 'href' in link.attrs:
            href = link['href']
            
            # Construct full URL
            if href.startswith('http'):
                product_url = href
            else:
                product_url = f"https://www.amazon.com{href}"
            
            # Clean URL (remove tracking params)
            product_url = product_url.split('?')[0] if '?' in product_url else product_url
            
            print(f"    ✓ Found product")
            return product_url
        
        print("    ✗ Could not extract product link")
        return None
        
    except Exception as e:
        print(f"    ✗ Search error: {type(e).__name__}")
        return None


def extract_amazon_price(soup):
    """Extract price from Amazon product page"""
    
    # Strategy 1: Whole price span
    price_whole = soup.find('span', {'class': 'a-price-whole'})
    price_fraction = soup.find('span', {'class': 'a-price-fraction'})
    
    if price_whole:
        whole = clean_price_text(price_whole.get_text())
        fraction = clean_price_text(price_fraction.get_text()) if price_fraction else '00'
        
        try:
            price = float(f"{whole}.{fraction}")
            if 0 < price < 1000000:
                return price
        except:
            pass
    
    # Strategy 2: Offscreen price (actual price for screen readers)
    offscreen = soup.find('span', {'class': 'a-offscreen'})
    if offscreen:
        price = clean_price_text(offscreen.get_text())
        if price:
            try:
                price_val = float(price)
                if 0 < price_val < 1000000:
                    return price_val
            except:
                pass
    
    # Strategy 3: Price block
    for selector in ['priceblock_ourprice', 'priceblock_dealprice', 'priceblock_saleprice']:
        elem = soup.find('span', {'id': selector})
        if elem:
            price = clean_price_text(elem.get_text())
            if price:
                try:
                    price_val = float(price)
                    if 0 < price_val < 1000000:
                        return price_val
                except:
                    pass
    
    # Strategy 4: Any element with price in class/id
    for elem in soup.find_all(['span', 'div', 'p']):
        classes = ' '.join(elem.get('class', [])).lower()
        elem_id = elem.get('id', '').lower()
        
        if 'price' in classes or 'price' in elem_id:
            text = elem.get_text().strip()
            if '$' in text or '₹' in text:
                price = clean_price_text(text)
                if price:
                    try:
                        price_val = float(price)
                        if 0 < price_val < 1000000:
                            return price_val
                    except:
                        pass
    
    return None


def clean_price_text(text):
    """Extract numeric price from text"""
    if not text:
        return None
    
    # Remove currency symbols and common words
    text = str(text).replace('$', '').replace('₹', '').replace('Rs', '').replace(',', '')
    
    # Extract numbers and decimal
    match = re.search(r'(\d+\.?\d*)', text)
    if match:
        return match.group(1)
    
    return None


def scrape_product_price(sample_id, catalog_content):
    """Main function to scrape price for a product"""
    print(f"\n[{sample_id}]")
    print(f"  Title: {catalog_content[:60]}...")
    
    # Search for product
    product_url = search_amazon_product(catalog_content)
    
    if not product_url:
        return None
    
    print(f"  URL: {product_url[:60]}...")
    
    # Scrape product page
    try:
        time.sleep(random.uniform(2, 4))
        
        response = requests.get(product_url, headers=HEADERS, timeout=15)
        
        if response.status_code != 200:
            print(f"  ✗ Page load failed: HTTP {response.status_code}")
            return None
        
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Extract price
        price = extract_amazon_price(soup)
        
        if price:
            print(f"  ✓ Price: ${price:.2f}")
            
            return {
                'sample_id': sample_id,
                'catalog_content': catalog_content,
                'product_url': product_url,
                'scraped_price': price
            }
        else:
            print("  ✗ Price not found on page")
            return None
            
    except Exception as e:
        print(f"  ✗ Scraping error: {type(e).__name__}: {str(e)[:50]}")
        return None


def main():
    print("\n" + "="*70)
    print("CATALOG-BASED PRICE SCRAPER")
    print("="*70)
    print(f"\n⚠️  WARNING: Scraping TEST data - DO NOT use for training! ⚠️")
    print(f"\nThis is for ANALYSIS ONLY. Using test data for training is CHEATING.")
    print("="*70)
    print(f"\nConfiguration:")
    print(f"  Test CSV: {TEST_CSV}")
    print(f"  Output:   {OUTPUT_DIR}")
    print(f"  Samples:  {MAX_SAMPLES if MAX_SAMPLES else 'ALL'}")
    print(f"  Method:   Amazon Direct Search")
    print("="*70)
    
    # Validate file
    if not os.path.exists(TEST_CSV):
        print(f"\n✗ ERROR: {TEST_CSV} not found!")
        return
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # Load data
    print(f"\nLoading test.csv...")
    df = pd.read_csv(TEST_CSV)
    print(f"✓ Loaded {len(df)} samples")
    
    # Validate columns
    if 'catalog_content' not in df.columns:
        print("\n✗ ERROR: 'catalog_content' column not found!")
        print(f"Available columns: {list(df.columns)}")
        return
    
    # Limit samples
    if MAX_SAMPLES:
        df = df.head(MAX_SAMPLES)
        print(f"Processing first {MAX_SAMPLES} samples")
    
    print("\n" + "="*70)
    print("STARTING SCRAPING...")
    print("="*70)
    
    # Scrape
    results = []
    success = 0
    fail = 0
    
    for idx, row in df.iterrows():
        result = scrape_product_price(row['sample_id'], row['catalog_content'])
        
        if result:
            results.append(result)
            success += 1
        else:
            fail += 1
        
        # Progress update
        if (idx + 1) % 5 == 0:
            print(f"\n{'─'*70}")
            print(f"Progress: {idx + 1}/{len(df)} | ✓ {success} | ✗ {fail} | Success Rate: {success/(idx+1)*100:.1f}%")
            print(f"{'─'*70}")
    
    # Final results
    print(f"\n{'='*70}")
    print("SCRAPING COMPLETE")
    print(f"{'='*70}")
    print(f"Total Processed:  {len(df)}")
    print(f"Successful:       {success} ({success/len(df)*100:.1f}%)")
    print(f"Failed:           {fail} ({fail/len(df)*100:.1f}%)")
    print(f"{'='*70}\n")
    
    # Save results
    if results:
        output_file = os.path.join(OUTPUT_DIR, 'test_prices_catalog_scraped.csv')
        results_df = pd.DataFrame(results)
        results_df.to_csv(output_file, index=False)
        
        print(f"✓ Results saved to: {output_file}\n")
        
        # Show preview
        print("Preview of scraped data:")
        print("─"*70)
        print(results_df.head(10).to_string())
        print("\n")
        
        # Statistics
        print("Price Statistics:")
        print(f"  Mean:   ${results_df['scraped_price'].mean():.2f}")
        print(f"  Median: ${results_df['scraped_price'].median():.2f}")
        print(f"  Min:    ${results_df['scraped_price'].min():.2f}")
        print(f"  Max:    ${results_df['scraped_price'].max():.2f}")
    else:
        print("✗ No results to save")
        print("\nTroubleshooting:")
        print("  - Check your internet connection")
        print("  - Amazon may be blocking automated requests")
        print("  - Try reducing MAX_SAMPLES and test with fewer items")
        print("  - Increase delay times in the code")
    
    print("\n" + "="*70)
    print("⚠️  CRITICAL REMINDER ⚠️")
    print("="*70)
    print("DO NOT use this scraped test data for training your model!")
    print("This would be data leakage and invalidate your results.")
    print("Use ONLY the original train.csv for model training.")
    print("="*70 + "\n")


if __name__ == '__main__':
    main()